# 02 — Rental Price Prediction

## Objective

Build a machine learning model to predict a car's daily rental price from its vehicle characteristics and equipment.

The final model will be saved as a complete preprocessing and prediction pipeline so that the same transformations can later be reused by the Getaround pricing API.

## Methodology

1. Load and validate the pricing dataset
2. Explore the target and model features
3. Prepare numerical and categorical variables
4. Establish a baseline regression model
5. Train and compare candidate regression models
6. Evaluate the selected model on unseen test data
7. Interpret the main drivers of predicted rental price
8. Save the final prediction pipeline for deployment

## 1. Setup and Data Loading

We first import the required libraries, define reproducible project paths, and load the pricing dataset.

The raw CSV is kept unchanged. Initial validation will determine which columns are useful for modeling and whether any exported index columns should be removed.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV


# ---------------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "get_around_pricing_project.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = OUTPUT_DIR / "models"
FIGURE_DIR = OUTPUT_DIR / "figures"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
display(df.head())

## 2. Data Structure and Quality Checks

Before modeling, we inspect feature types, missing values, duplicates, and identifier-like columns.

This step ensures that the modeling dataset contains meaningful predictive features and that no unnecessary exported index is used by the model.

In [ ]:
# ---------------------------------------------------------------------------
# Dataset structure
# ---------------------------------------------------------------------------

print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(
    df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_pct=lambda x: x["missing_count"] / len(df) * 100)
)

print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"Unique values in 'Unnamed: 0': {df['Unnamed: 0'].nunique():,}")
print(f"Rows in dataset: {len(df):,}")

## 3. Modeling Dataset Preparation

The `Unnamed: 0` column is an exported row index: it is unique for every observation and does not represent a vehicle characteristic. It is therefore excluded from modeling.

The remaining columns contain vehicle characteristics and equipment features, with `rental_price_per_day` as the regression target.

In [ ]:
# ---------------------------------------------------------------------------
# Remove non-predictive exported index
# ---------------------------------------------------------------------------

df_model = df.drop(columns=["Unnamed: 0"]).copy()

TARGET = "rental_price_per_day"

print(f"Modeling dataset shape: {df_model.shape}")
print(f"Target: {TARGET}")

display(df_model.head())

## 4. Target Variable Analysis

Before training a regression model, we examine the distribution of `rental_price_per_day`.

Understanding the target's range, central tendency, and potential extreme values helps us choose appropriate evaluation metrics and interpret model performance later.

In [ ]:
# ---------------------------------------------------------------------------
# Target summary
# ---------------------------------------------------------------------------

target_summary = df_model[TARGET].describe()

display(target_summary.to_frame("rental_price_per_day"))


### Rental Price Distribution

The target distribution is visualized to assess its overall shape and identify whether unusually high or low rental prices may influence regression performance.

In [ ]:
# ---------------------------------------------------------------------------
# Target distribution
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(
    df_model[TARGET],
    bins=40,
    edgecolor="black",
    alpha=0.8,
)

ax.axvline(
    df_model[TARGET].median(),
    linestyle="--",
    label=f"Median = €{df_model[TARGET].median():.0f}",
)

ax.set_title("Distribution of Daily Rental Prices")
ax.set_xlabel("Rental price per day (€)")
ax.set_ylabel("Number of cars")
ax.legend()

plt.tight_layout()
plt.show()

### Interpretation

Daily rental prices are concentrated around the center of the distribution, with a median of **€119** and a mean of approximately **€121**.

The distribution is moderately right-skewed, with a small number of higher-priced vehicles extending beyond the main concentration of observations. These values are retained because they may represent legitimate premium vehicles rather than data errors.

For model evaluation, we will therefore use metrics such as **MAE**, which provides an intuitive average prediction error in euros, together with **RMSE** and **R²** for a broader assessment of predictive performance.

## 5. Feature Overview

The predictors include numerical vehicle characteristics, categorical attributes, and binary equipment indicators.

Before building the preprocessing pipeline, we inspect the number of unique values in each feature. This helps identify which variables require numerical scaling, categorical encoding, or simple binary handling.

In [ ]:
# ---------------------------------------------------------------------------
# Feature overview
# ---------------------------------------------------------------------------

X = df_model.drop(columns=[TARGET])

feature_overview = pd.DataFrame(
    {
        "dtype": X.dtypes.astype(str),
        "unique_values": X.nunique(),
    }
).sort_values("unique_values")

display(feature_overview)

## 6. Train/Test Split

The dataset is divided into training and test sets before fitting any preprocessing steps.

The training set will be used for preprocessing and model development, while the test set will remain unseen until final evaluation. A fixed random seed ensures reproducibility.

In [ ]:

# ---------------------------------------------------------------------------
# Features and target
# ---------------------------------------------------------------------------

X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

# ---------------------------------------------------------------------------
# Train/test split
# ---------------------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")

## 7. Preprocessing Pipeline

The model contains three types of predictors:

- **Numerical features:** mileage and engine power
- **Categorical features:** vehicle model, fuel type, paint color, and car type
- **Boolean features:** parking availability and vehicle equipment indicators

Numerical features are standardized, categorical features are one-hot encoded, and boolean features are passed through unchanged.

All preprocessing is defined inside a `ColumnTransformer`. This ensures that transformations are learned only from the training data and can later be packaged with the prediction model for deployment.

In [ ]:
# ---------------------------------------------------------------------------
# Feature groups
# ---------------------------------------------------------------------------

numeric_features = [
    "mileage",
    "engine_power",
]

categorical_features = [
    "model_key",
    "fuel",
    "paint_color",
    "car_type",
]

boolean_features = [
    "private_parking_available",
    "has_gps",
    "has_air_conditioning",
    "automatic_car",
    "has_getaround_connect",
    "has_speed_regulator",
    "winter_tires",
]

# ---------------------------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        ("boolean", "passthrough", boolean_features),
    ]
)

print("Preprocessing pipeline ready.")

## 8. Baseline Model — Linear Regression

A linear regression model is used as the first predictive baseline.

The preprocessing transformer and regression model are combined in a single scikit-learn `Pipeline`. This ensures that exactly the same preprocessing steps are applied during training, evaluation, and later API inference.

Performance is evaluated using **MAE**, **RMSE**, and **R²**.

In [ ]:
# ---------------------------------------------------------------------------
# Baseline pipeline
# ---------------------------------------------------------------------------

linear_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

linear_pipeline.fit(X_train, y_train)

# ---------------------------------------------------------------------------
# Test predictions
# ---------------------------------------------------------------------------

y_pred_linear = linear_pipeline.predict(X_test)

linear_mae = mean_absolute_error(y_test, y_pred_linear)
linear_rmse = np.sqrt(mean_squared_error(y_test, y_pred_linear))
linear_r2 = r2_score(y_test, y_pred_linear)

print(f"MAE:  €{linear_mae:.2f}")
print(f"RMSE: €{linear_rmse:.2f}")
print(f"R²:   {linear_r2:.3f}")

### Baseline Interpretation

The linear regression provides a solid baseline:

- **MAE: €12.12** — predictions differ from observed daily rental prices by about €12 on average.
- **RMSE: €17.96** — larger prediction errors increase this metric relative to MAE.
- **R²: 0.694** — the model explains approximately 69% of the variation in daily rental prices in the test set.

This indicates that the available vehicle characteristics contain substantial predictive information, while leaving room for improvement with a model capable of capturing nonlinear relationships and interactions between features.

## 9. Regularized Linear Regression

The baseline linear regression does not regularize its coefficients. After
one-hot encoding categorical variables, the feature space contains many
correlated indicator variables, making regularization a useful extension.

Ridge regression adds L2 regularization to reduce overly large coefficients.
The regularization strength (`alpha`) is selected using cross-validation on
the training data, while the test set remains untouched.

In [ ]:

# ---------------------------------------------------------------------------
# Ridge regression with cross-validation
# ---------------------------------------------------------------------------

ridge_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0]),
        ),
    ]
)

ridge_pipeline.fit(X_train, y_train)

y_pred_ridge = ridge_pipeline.predict(X_test)

ridge_mae = mean_absolute_error(y_test, y_pred_ridge)
ridge_rmse = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
ridge_r2 = r2_score(y_test, y_pred_ridge)

best_alpha = ridge_pipeline.named_steps["model"].alpha_

print(f"Best alpha: {best_alpha}")
print(f"MAE:  €{ridge_mae:.2f}")
print(f"RMSE: €{ridge_rmse:.2f}")
print(f"R²:   {ridge_r2:.3f}")

### Regularization Result

Cross-validated Ridge regression selected an alpha of **1.0**, but its test
performance was essentially identical to the unregularized linear regression
(MAE €12.12, RMSE €17.97, R² 0.693).

Regularization therefore does not provide a meaningful improvement for this
dataset. The original linear regression remains the simpler linear baseline,
and the next step is to test whether a nonlinear model can capture relationships
that the linear specification misses.

## 10. Nonlinear Model — Random Forest

Linear models assume that feature effects combine primarily in an additive,
linear way. Rental prices may instead depend on nonlinear relationships and
interactions between vehicle characteristics.

A Random Forest regressor is therefore tested as a nonlinear alternative.
It can model more complex relationships while remaining relatively robust
and interpretable through feature importance.

The same train/test split and preprocessing framework are retained to ensure
a fair comparison with the linear baseline.

In [ ]:

# ---------------------------------------------------------------------------
# Random Forest pipeline
# ---------------------------------------------------------------------------

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

rf_pipeline.fit(X_train, y_train)

# ---------------------------------------------------------------------------
# Test predictions
# ---------------------------------------------------------------------------

y_pred_rf = rf_pipeline.predict(X_test)

rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2 = r2_score(y_test, y_pred_rf)

print(f"MAE:  €{rf_mae:.2f}")
print(f"RMSE: €{rf_rmse:.2f}")
print(f"R²:   {rf_r2:.3f}")

### Random Forest Result

The Random Forest improves predictive performance over both linear models:

- **MAE decreases from €12.12 to €10.63**, an improvement of approximately 12%.
- **RMSE decreases from €17.96 to €16.71**, indicating better performance on larger errors as well.
- **R² increases from 0.694 to 0.735**, meaning the model explains a larger share of the variation in unseen rental prices.

These results suggest that nonlinear relationships and interactions between
vehicle characteristics contain useful predictive information.

Random Forest is therefore the strongest model tested so far. Before selecting
the final model, a small amount of hyperparameter tuning will be performed
using cross-validation on the training set only.

## 11. Random Forest Hyperparameter Tuning

The initial Random Forest outperformed the linear models. A small
hyperparameter search is therefore performed to determine whether its
performance can be improved further.

Cross-validation is performed only on the training set. The search is kept
deliberately compact to balance model improvement, computational cost, and
project complexity.

In [ ]:

# ---------------------------------------------------------------------------
# Small Random Forest parameter grid
# ---------------------------------------------------------------------------

param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 15],
    "model__min_samples_leaf": [1, 2, 4],
}

rf_search_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

grid_search = GridSearchCV(
    estimator=rf_search_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print(
    f"\nBest cross-validation MAE: "
    f"€{-grid_search.best_score_:.2f}"
)

### Tuned Random Forest Evaluation

The hyperparameter search selected the model configuration with the lowest
cross-validated MAE on the training data.

The selected model is now evaluated once on the untouched test set using
MAE, RMSE, and R². This provides a direct comparison with the untuned
Random Forest and the linear baselines.

In [ ]:
# ---------------------------------------------------------------------------
# Evaluate tuned Random Forest on test set
# ---------------------------------------------------------------------------

best_rf_pipeline = grid_search.best_estimator_

y_pred_best_rf = best_rf_pipeline.predict(X_test)

best_rf_mae = mean_absolute_error(y_test, y_pred_best_rf)
best_rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_best_rf))
best_rf_r2 = r2_score(y_test, y_pred_best_rf)

print(f"MAE:  €{best_rf_mae:.2f}")
print(f"RMSE: €{best_rf_rmse:.2f}")
print(f"R²:   {best_rf_r2:.3f}")

### Tuning Result

The tuned Random Forest achieved the strongest test performance:

- **MAE: €10.59**
- **RMSE: €16.57**
- **R²: 0.739**

Compared with the untuned Random Forest, hyperparameter tuning provides a
small additional improvement. The larger gain came from moving from a linear
model to a nonlinear ensemble model.

The tuned Random Forest is therefore retained as the final candidate model.

## 12. Final Model Diagnostics

The tuned Random Forest is evaluated beyond aggregate metrics by comparing
predicted rental prices with their observed values.

If predictions were perfect, all observations would lie on the diagonal
reference line. Deviations from this line reveal where the model tends to
overestimate or underestimate rental prices.

In [ ]:
# ---------------------------------------------------------------------------
# Predicted vs. actual prices
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(
    y_test,
    y_pred_best_rf,
    alpha=0.5,
)

min_price = min(y_test.min(), y_pred_best_rf.min())
max_price = max(y_test.max(), y_pred_best_rf.max())

ax.plot(
    [min_price, max_price],
    [min_price, max_price],
    linestyle="--",
)

ax.set_title("Predicted vs. Actual Rental Prices")
ax.set_xlabel("Actual rental price per day (€)")
ax.set_ylabel("Predicted rental price per day (€)")

plt.tight_layout()
plt.show()

### Predicted vs. Actual Interpretation

Predictions generally follow the diagonal reference line, particularly across
the main concentration of rental prices.

The model shows larger errors for some observations at the extremes of the
price distribution. In particular, very high rental prices can be
underestimated, while some very low-priced vehicles are overestimated.

This suggests that the model captures the general pricing structure well but
is less accurate for unusual or extreme vehicles, which are relatively rare
in the dataset.

### Residual Analysis

Residuals represent the difference between the observed and predicted rental
prices. Plotting residuals against predicted prices helps identify systematic
prediction errors.

Ideally, residuals should be distributed around zero without a strong pattern.

In [ ]:
# ---------------------------------------------------------------------------
# Residual analysis
# ---------------------------------------------------------------------------

residuals = y_test - y_pred_best_rf

fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(
    y_pred_best_rf,
    residuals,
    alpha=0.5,
)

ax.axhline(
    0,
    linestyle="--",
)

ax.set_title("Residuals vs. Predicted Rental Prices")
ax.set_xlabel("Predicted rental price per day (€)")
ax.set_ylabel("Residual (€)")

plt.tight_layout()
plt.show()

### Residual Interpretation

Residuals are generally distributed around zero without a strong systematic
pattern, suggesting that the model captures the main pricing relationships
reasonably well.

Most prediction errors are relatively concentrated, although several larger
residuals remain. These errors are consistent with the model's weaker
performance on unusual or extreme rental prices.

Overall, the diagnostics support using the tuned Random Forest as the selected
model, while acknowledging that predictions for atypical vehicles should be
interpreted with greater caution.

## 13. Model Interpretation — Feature Importance

Random Forest feature importance provides a high-level view of which input
variables contribute most to the model's predictions.

Because categorical variables are one-hot encoded, individual categories
appear as separate transformed features. The importance values describe the
model's predictive reliance on each feature; they should not be interpreted
as causal effects on rental prices.

In [ ]:
# ---------------------------------------------------------------------------
# Extract transformed feature names and Random Forest importances
# ---------------------------------------------------------------------------

fitted_preprocessor = best_rf_pipeline.named_steps["preprocessor"]
fitted_model = best_rf_pipeline.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()
feature_importances = fitted_model.feature_importances_

importance_df = (
    pd.DataFrame(
        {
            "feature": feature_names,
            "importance": feature_importances,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(15))

### Most Important Predictive Features

The chart below displays the 15 most important transformed features used by
the selected Random Forest model.

Importance reflects how much the model relies on each feature for prediction.
It does not indicate whether a feature increases or decreases rental price,
and it should not be interpreted causally.

In [ ]:
# ---------------------------------------------------------------------------
# Plot top feature importances
# ---------------------------------------------------------------------------

top_importances = importance_df.head(15).sort_values("importance")

fig, ax = plt.subplots(figsize=(8, 6))

ax.barh(
    top_importances["feature"],
    top_importances["importance"],
)

ax.set_title("Top 15 Random Forest Feature Importances")
ax.set_xlabel("Feature importance")
ax.set_ylabel("Feature")

plt.tight_layout()
plt.show()

### Feature Importance Interpretation

**Engine power** and **mileage** are the dominant predictive features in the
selected model, together accounting for most of the Random Forest's
impurity-based feature importance.

Equipment and vehicle characteristics such as **GPS**, **Getaround Connect**,
car type, brand, and transmission also contribute to predictions, but with
smaller individual importance values.

These results indicate which variables the model relies on most for prediction.
They do not establish causal relationships or indicate the direction of a
feature's effect on rental price.

## 14. Save the Final Prediction Pipeline

The selected model is saved as a single fitted scikit-learn pipeline containing
both preprocessing and the tuned Random Forest regressor.

Saving the complete pipeline ensures that future API requests receive exactly
the same feature transformations used during model training.

In [ ]:
# ---------------------------------------------------------------------------
# Save fitted preprocessing + model pipeline
# ---------------------------------------------------------------------------

MODEL_PATH = MODEL_DIR / "getaround_pricing_model.joblib"

joblib.dump(best_rf_pipeline, MODEL_PATH)

print(f"Model saved to: {MODEL_PATH}")

### Deployment Artifact Validation

Before deployment, the saved pipeline is reloaded and used to generate
predictions on the test data.

The reloaded model should produce the same predictions as the in-memory
pipeline, confirming that the serialized artifact is ready for use by the API.

In [ ]:
# ---------------------------------------------------------------------------
# Reload and validate saved pipeline
# ---------------------------------------------------------------------------

loaded_pipeline = joblib.load(MODEL_PATH)

reloaded_predictions = loaded_pipeline.predict(X_test)

predictions_match = np.allclose(
    y_pred_best_rf,
    reloaded_predictions,
)

print(f"Reloaded predictions match: {predictions_match}")

## 15. Conclusion

This notebook developed and evaluated a regression pipeline for predicting
Getaround daily rental prices from vehicle characteristics.

The linear regression baseline achieved an **MAE of €12.12** and an
**R² of 0.694**. Ridge regularization did not provide a meaningful improvement.

A Random Forest captured additional nonlinear relationships in the data.
After a compact cross-validation search, the selected Random Forest achieved:

- **MAE: €10.59**
- **RMSE: €16.57**
- **R²: 0.739**

Model diagnostics showed generally good agreement across the main rental-price
range, with larger errors for some unusual or extreme-priced vehicles.

Feature importance indicated that **engine power** and **mileage** were the
strongest predictive variables, while vehicle equipment, model, and car type
provided additional information. These importances describe predictive
relevance and should not be interpreted causally.

The complete fitted preprocessing and prediction pipeline was saved as
`outputs/models/getaround_pricing_model.joblib` and successfully reloaded with
identical predictions. It is therefore ready to be integrated into the
pricing prediction API.